In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from olbootstrap.online_bootstrap._block_bootstrap import BlockBootstrap
from olbootstrap.online_bootstrap._online_ar_bootstrap import OnlineARBootstrap
from olbootstrap.online_bootstrap._online_gaussian_bootstrap import (
    OnlineGaussianMixtureAsympCSSmoothedBootstrap,
)
from olbootstrap.synthetic_time_series._ar1 import AR1Process

In [ ]:
def _runtime_K(sample_size: int, burn_in: int, var_warmup: int) -> int:
    """Match the dyadic K used in your experiments."""
    if var_warmup <= 0:
        return 1
    return int(max(1, np.ceil(np.log2((sample_size - burn_in) / float(var_warmup)))))


def make_runtime_bootstrap(
    *,
    method: str,
    B: int,
    eta: float,
    sample_size: int,
    burn_in: int,
    var_warmup: int,
    alpha: float,
    rng: np.random.Generator,
    smoothing_method: str = "ewma",
):
    """Construct one bootstrap object for update-time benchmarking."""
    K = _runtime_K(
        sample_size=sample_size,
        burn_in=burn_in,
        var_warmup=var_warmup,
    )

    if method == "ours":
        return OnlineARBootstrap(
            smoothing_method=smoothing_method,
            eta=eta,
            var_warmup=var_warmup,
            use_variance_smoothing=False,
            smoothing_beta=None,
            gamma=None,
            seasonal_period=None,
            forecast_s=0,
            rng=rng,
            transform="student",
            transform_power=1.0 / 3.0,
            rho_power=-1.0 / 3.0,
            K=K,
            t0=burn_in,
            alpha=alpha,
        )

    if method == "iid":
        return OnlineARBootstrap(
            smoothing_method=smoothing_method,
            eta=eta,
            var_warmup=var_warmup,
            use_variance_smoothing=False,
            smoothing_beta=None,
            gamma=None,
            seasonal_period=None,
            forecast_s=0,
            rng=rng,
            transform="gauss",
            transform_power=1.0 / 3.0,
            rho_power=0.0,
            K=K,
            t0=burn_in,
            alpha=alpha,
        )

    if method == "WS":
        return OnlineGaussianMixtureAsympCSSmoothedBootstrap(
            smoothing_method=smoothing_method,
            eta=eta,
            var_warmup=var_warmup,
            use_variance_smoothing=False,
            smoothing_beta=None,
            gamma=None,
            seasonal_period=None,
            forecast_s=0,
            rng=rng,
            transform="student",
            transform_power=1.0 / 3.0,
            K=K,
            t0=burn_in,
            alpha=alpha,
        )

    if method == "block":
        return BlockBootstrap(
            smoothing_method=smoothing_method,
            eta=eta,
            var_warmup=var_warmup,
            use_variance_smoothing=False,
            smoothing_beta=None,
            gamma=None,
            seasonal_period=None,
            forecast_s=0,
            rng=rng,
            transform="student",
            transform_power=1.0 / 3.0,
            K=K,
            t0=burn_in,
            alpha=alpha,
            sample_size=sample_size,
            recompute_every=1,
            uniform=True,
        )

    raise ValueError(f"Unknown method: {method}")

In [ ]:
def benchmark_update_times(
    *,
    samples: np.ndarray,
    methods: tuple[str, ...] = ("ours", "iid", "WS", "block"),
    B: int = 200,
    eta: float = 2 / 100,
    burn_in: int = 500,
    var_warmup: int = 400,
    alpha: float = 0.1,
    smoothing_method: str = "ewma",
    seed: int = 12345,
    repeat: int = 0,
) -> pd.DataFrame:
    """Measure wall-clock time of each individual bootstrap update.

    The measured quantity is the time for one direct call to the bootstrap
    object:

        bootstrap(np.array([x_t]))

    This excludes data generation, interval post-processing, coverage
    computation, plotting, and pandas aggregation.
    """
    samples = np.asarray(samples, dtype=float).reshape(-1)
    n = samples.size

    # Pre-create one-sample arrays so array allocation outside the bootstrap
    # call is not part of the measured time.
    sample_arrays = [np.array([x], dtype=float) for x in samples]

    rows = []

    for method_idx, method in enumerate(methods):
        rng = np.random.default_rng(seed + 10_000 * repeat + 100 * method_idx)

        bootstrap = make_runtime_bootstrap(
            method=method,
            B=B,
            eta=eta,
            sample_size=n,
            burn_in=burn_in,
            var_warmup=var_warmup,
            alpha=alpha,
            rng=rng,
            smoothing_method=smoothing_method,
        )

        update_times = np.empty(n, dtype=float)

        for t, x_arr in enumerate(sample_arrays, start=1):
            start = time.perf_counter()

            if t == 1:
                bootstrap(x_arr, number_bootstrap_samples=B)
            else:
                bootstrap(x_arr)

            update_times[t - 1] = time.perf_counter() - start

        rows.append(
            pd.DataFrame(
                {
                    "method": method,
                    "t": np.arange(1, n + 1, dtype=int),
                    "update_time_sec": update_times,
                    "eta": float(eta),
                    "B": int(B),
                    "n": int(n),
                    "burn_in": int(burn_in),
                    "var_warmup": int(var_warmup),
                    "burn_eff": int(burn_in + var_warmup),
                    "repeat": int(repeat),
                }
            )
        )

    return pd.concat(rows, ignore_index=True)

In [ ]:
rng = np.random.default_rng(1234)

proc = AR1Process(
    mean=0.0,
    phi=0.3,
    noise_std=1.0,
    trend_slope=0.0,
    rng=rng,
)

sample_size = 5000
samples = proc.generate_samples(sample_size)

runtime_updates = benchmark_update_times(
    samples=samples,
    methods=("ours", "iid", "WS", "block"),
    B=400,
    eta=2 / 100,
    burn_in=0,
    var_warmup=0,
    alpha=0.1,
    smoothing_method="ewma",
    seed=12345,
    repeat=0,
)

In [ ]:
def add_rolling_update_time(
    df: pd.DataFrame,
    *,
    window: int = 101,
    time_col: str = "update_time_sec",
) -> pd.DataFrame:
    """Add rolling median update time per method/repeat."""
    out = df.sort_values(["method", "repeat", "t"]).copy()

    out["update_time_rollmed_sec"] = (
        out.groupby(["method", "repeat"], group_keys=False)[time_col]
        .rolling(window=window, center=True, min_periods=max(5, window // 10))
        .median()
        .reset_index(level=[0, 1], drop=True)
    )

    return out


runtime_updates_smooth = add_rolling_update_time(
    runtime_updates,
    window=101,
)

In [ ]:
import matplotlib.pyplot as plt

outdir = Path("figs")
outdir.mkdir(parents=True, exist_ok=True)
save_path = outdir / "runtime_per_update_over_time.pdf"

df_plot = runtime_updates_smooth.copy()

fig, ax = plt.subplots(figsize=(6.0, 3.6), dpi=250)

method_display = {
    "ours": "ours",
    "iid": "iid",
    "WS": "WS",
    "block": "block",
}

method_colors = {
    "ours": "#0C5DA5",  # SciencePlots blue
    "iid": "#00B945",  # SciencePlots green
    "WS": "#FF9500",  # SciencePlots orange
    "block": "#B07AA1",  # SciencePlots violet
}

method_order = ["ours", "iid", "WS", "block"]

for method in method_order:
    dd = df_plot[df_plot["method"] == method].copy()
    if dd.empty:
        continue
    dd = dd.sort_values("t")

    ax.plot(
        dd["t"],
        dd["update_time_rollmed_sec"],
        linewidth=1.6,
        color=method_colors[method],
        label=method_display.get(method, method),
    )

burn_eff = int(df_plot["burn_eff"].iloc[0])
ax.axvline(
    burn_eff,
    linestyle="--",
    linewidth=1.2,
    color="gray",
    alpha=0.85,
)

ax.set_xlabel(r"Time index $t$", fontsize=16)
ax.set_ylabel("Time per update (seconds)", fontsize=16)
ax.set_yscale("log")

ax.set_xlim(0, 5000)
ax.margins(x=0)

ax.grid(True, linestyle="--", alpha=0.5)
ax.tick_params(axis="both", which="major", labelsize=14)
ax.legend(fontsize=10, frameon=True, loc="lower right")


fig.tight_layout(pad=0.6)
fig.savefig(save_path, bbox_inches="tight")
plt.show()

print("Saved to:", save_path.resolve())